# L12：真实天气接口——日期如何影响工具调用

<!-- course-meta: real-llm -->

分别询问广州天河区昨天、明天和今天的天气，观察模型选择的工具和传入的时间窗口，再改善“今天”的处理规则。这里调用真实天气服务和真实模型，不使用离线天气数据或 `COURSE_NOW`。

运行前请确保 `.env` 配置了 `BASE_URL_WEATHER_API`、`MAIN_API_KEY`，且网络能访问天气服务。天气查询直接传地点名称。接口参数按原项目约定使用北京时。

In [1]:
from datetime import datetime, timedelta
from langchain.agents import create_agent
from common import llm
from common.trace import show, tool_calls
from lessons_import import load_lesson_module

weather = load_lesson_module("L12_天气查询专线", "real_weather_tools")
TOOLS = [weather.get_current_beijing_time, weather.get_station_history, weather.get_forecast]
model = llm.main_llm()
print("真实当前北京时：", weather.get_current_beijing_time.invoke({}))

真实当前北京时： 2026-10-29 17:37:01


## 1. 先预测，再实验

| 提问 | 需要什么数据？ | 应当覆盖的北京时段 |
|---|---|---|
| 昨天 | 实况 | 昨天 00:00 至 23:59 |
| 明天 | 预报 | 明天 00:00 至 23:59 |
| 今天 | 用户需要的是实况还是预报呢？ | 今天 00:00 至现在；现在至今天结束 |

预报仅覆盖模型起报后时段，站点实况也可能有更新延迟。接口没有数据或覆盖不全时，应如实说明。运行期间跨日时，应以工具实际返回的当前时间重新判断。

In [2]:
BASE_PROMPT = """你是天气查询助手，用户说的时间均为北京时。
"""

def run_question(question, system_prompt=BASE_PROMPT):
    agent = create_agent(model=model, system_prompt=system_prompt, tools=TOOLS)
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    show(result, max_tool_chars=600)
    calls = tool_calls(result)
    print("天气工具及其参数：")
    for call in calls:
        if call["name"] in ("get_station_history", "get_forecast"):
            print(" ", call["name"], call["args"])
    return result

## 2. 昨天：历史实况

先预测：要查哪个工具？开始和结束日期是什么？运行后核对调用记录、数据覆盖和回答。

In [3]:
yesterday_result = run_question("广州天河区昨天的天气如何？")

[1] 用户：广州天河区昨天的天气如何？

[2] 模型决定调用工具 get_current_beijing_time，参数 {}

[3] 工具 get_current_beijing_time 返回：2026-10-29 17:38:15

[4] 模型决定调用工具 get_station_history，参数 {'address': '广州天河区', 'start_time': '2026-10-28 00:00:00', 'end_time': '2026-10-28 23:59:00'}

[5] 工具 get_station_history 返回：{"status": "ok", "data": {"广州天河区": [{"datetime": "2026-10-28 00:00:00", "rain": 0.0, "wind_vel": 0.5, "wind_dir": 318.0, "gust": 0.8, "t2mm": 25.1}, {"datetime": "2026-10-28 01:00:00", "rain": 0.0, "wind_vel": 0.6, "wind_dir": 339.0, "gust": 1.3, "t2mm": 25.1}, {"datetime": "2026-10-28 02:00:00", "rain": 0.0, "wind_vel": 0.7, "wind_dir": 292.0, "gust": 0.9, "t2mm": 25.3}, {"datetime": "2026-10-28 03:00:00", "rain": 0.0, "wind_vel": 0.9, "wind_dir": 286.0, "gust": 1.7, "t2mm": 25.2}, {"datetime": "2026-10-28 04:00:00", "rain": 0.0, "wind_vel": 0.8, "wind_dir": 314.0, "gust": 1.1, "t2mm": 24.4},……（共 2745 字，已截断）

[6] 模型回答：广州天河区昨天（2026年10月28日）天气总体**晴暖、干燥，全天无降水**。

- **气温**：约 **24.1℃～30.9℃**，清晨最低，下午14—15时最高
- **降

## 3. 明天：天气预报

先预测：要查哪个工具？预报工具参数中的查询时段与起报时次有什么区别？运行后核对工具调用和结果。

In [4]:
tomorrow_result = run_question("广州天河区明天的天气如何？")

[1] 用户：广州天河区明天的天气如何？

[2] 模型决定调用工具 get_current_beijing_time，参数 {}

[3] 工具 get_current_beijing_time 返回：2026-10-29 17:38:44

[4] 模型决定调用工具 get_forecast，参数 {'address': '广州天河区', 'start_time': '2026-10-30 00:00:00', 'end_time': '2026-10-30 23:59:59'}

[5] 工具 get_forecast 返回：{"status": "ok", "data": {"广州天河区": [{"datetime": "2026-10-30 00:00:00", "rain": 0.0, "t2mm": 26.15, "visi": 14.26, "wind_vel": 1.79, "wind_dir": 159.63, "gust": 3.4}, {"datetime": "2026-10-30 01:00:00", "rain": 0.0, "t2mm": 25.85, "visi": 13.75, "wind_vel": 1.51, "wind_dir": 152.16, "gust": 3.08}, {"datetime": "2026-10-30 02:00:00", "rain": 0.0, "t2mm": 25.55, "visi": 13.25, "wind_vel": 1.34, "wind_dir": 140.43, "gust": 2.87}, {"datetime": "2026-10-30 03:00:00", "rain": 0.0, "t2mm": 25.26, "visi": 12.63, "wind_vel": 1.33, "wind_dir": 114.77, "gust": 2.86}, {"datetime": "2026-10-30 04:00:00", "……（共 3251 字，已截断）

[6] 模型回答：广州天河区明天（10月30日）以晴到多云为主，全天无明显降水，体感偏热。

- **气温**：最低约 **24.7℃**（凌晨5时前后），最高约 **32.2℃**（下午2时前后）
- **降水**：全天降雨

## 4. 今天：实况与预报

先预测：今天的时间段与当前时间是什么关系？基础提示词没有明确拆分规则，观察模型是否调用了两个工具，以及窗口是否覆盖今天。实际数据有延迟或缺测时，要在回答中说明。

In [8]:
today_result = run_question("广州天河区现在出门需要防晒吗？")

[1] 用户：广州天河区现在出门需要防晒吗？

[2] 模型决定调用工具 get_current_beijing_time，参数 {}

[3] 工具 get_current_beijing_time 返回：2026-10-29 17:42:02

[4] 模型决定调用工具 get_station_history，参数 {'address': '广州天河区', 'start_time': '2026-10-29 14:00:00', 'end_time': '2026-10-29 17:42:00'}
[4] 模型决定调用工具 get_forecast，参数 {'address': '广州天河区', 'start_time': '2026-10-29 17:42:00', 'end_time': '2026-10-29 20:00:00'}

[5] 工具 get_station_history 返回：{"status": "ok", "data": {"广州天河区": [{"datetime": "2026-10-29 14:00:00", "rain": 0.0, "wind_vel": 0.9, "wind_dir": 317.0, "gust": 1.3, "t2mm": 30.4}, {"datetime": "2026-10-29 15:00:00", "rain": 0.0, "wind_vel": 0.9, "wind_dir": 14.0, "gust": 1.5, "t2mm": 29.6}, {"datetime": "2026-10-29 16:00:00", "rain": 0.0, "wind_vel": 0.8, "wind_dir": 351.0, "gust": 1.4, "t2mm": 28.9}, {"datetime": "2026-10-29 17:00:00", "rain": 0.0, "wind_vel": 0.7, "wind_dir": 341.0, "gust": 0.9, "t2mm": 27.8}]}}

[6] 工具 get_forecast 返回：{"status": "ok", "data": {"广州天河区": [{"datetime": "2026-10-29 17:42:00", "rain": 

## 5. 改进提示词，再问今天

明确跨越当前时间的拆分规则后，重新提问。比较两次的工具种类、起止时间和回答；基础版如果已经正确，也可以重复运行观察模型的稳定性。

In [9]:
IMPROVED_PROMPT = BASE_PROMPT + """
先依据当前北京时判断整个查询时段。过去部分调用 get_station_history，未来部分调用 get_forecast。
如果查询跨越当前时间（如今天），必须分别查询从当天 00:00 到现在的实况、从现在到当天结束的预报。
回答分为截至目前的实况和接下来的预报，并说明数据缺失、更新延迟或未覆盖时段。"""
today_improved = run_question("广州天河区今天的天气如何？", IMPROVED_PROMPT)

[1] 用户：广州天河区今天的天气如何？

[2] 模型决定调用工具 get_current_beijing_time，参数 {}

[3] 工具 get_current_beijing_time 返回：2026-10-29 17:42:56

[4] 模型决定调用工具 get_station_history，参数 {'address': '广州天河区', 'start_time': '2026-10-29 00:00:00', 'end_time': '2026-10-29 17:42:56'}
[4] 模型决定调用工具 get_forecast，参数 {'address': '广州天河区', 'start_time': '2026-10-29 17:42:56', 'end_time': '2026-10-29 23:59:59'}

[5] 工具 get_station_history 返回：{"status": "ok", "data": {"广州天河区": [{"datetime": "2026-10-29 00:00:00", "rain": 0.0, "wind_vel": 1.6, "wind_dir": 89.0, "gust": 3.2, "t2mm": 26.9}, {"datetime": "2026-10-29 01:00:00", "rain": 0.0, "wind_vel": 2.2, "wind_dir": 87.0, "gust": 3.4, "t2mm": 26.7}, {"datetime": "2026-10-29 02:00:00", "rain": 0.0, "wind_vel": 1.4, "wind_dir": 99.0, "gust": 2.2, "t2mm": 26.2}, {"datetime": "2026-10-29 03:00:00", "rain": 0.0, "wind_vel": 0.8, "wind_dir": 253.0, "gust": 1.6, "t2mm": 26.0}, {"datetime": "2026-10-29 04:00:00", "rain": 0.0, "wind_vel": 1.0, "wind_dir": 161.0, "gust": 1.8, "t2mm": 25.4

## 6. 比较调用流程

下表展示模型发起的调用，不能单凭调用过工具就认定数据齐全。请对照每次取到的当前时间和工具返回，检查具体时段、实况与预报的衔接、是否遗漏及回答是否忠实于数据。

In [ ]:
for label, result in [("昨天", yesterday_result), ("明天", tomorrow_result),
                      ("今天（基础）", today_result), ("今天（改进）", today_improved)]:
    calls = tool_calls(result)
    weather_calls = [c for c in calls if c["name"] in ("get_station_history", "get_forecast")]
    print(f"\n{label}：")
    for call in weather_calls:
        args = call["args"]
        print(f"  {call['name']}：{args.get('start_time')} ～ {args.get('end_time')}（北京时）")
    if not weather_calls:
        print("  未调用天气工具")

## 7. 动手练习

1. 将地点换成服务支持的其他地点，重新运行三个问题，核对日期和工具选择。
2. 问“昨天晚上”和“明天上午”，检查时间窗口是否准确，是否超出了用户要求。
3. 重复问“今天”相关的问题若干次，记录是否每次都覆盖实况与预报；实况和预报的时间参数有没有出错，尝试调整提示词并比较结果。